# SpaceX Falcon 9: Data Collection and Wrangling

**Goal:** prepare a dataset to predict whether the Falcon 9 first stage lands successfully.

**Data note:** the live SpaceX API (`api.spacexdata.com`) was unavailable when this was built (Cloudflare errors 522/525), so its ID lookups could not run. This notebook uses IBM's static snapshot of the API response and IBM's published result of the collection step (`dataset_part_1.csv`).

In [1]:
import pandas as pd
import requests

## 1. Request and parse the launch data

In [2]:
static_json_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
response = requests.get(static_json_url, timeout=60)
print(response.status_code)

data = pd.json_normalize(response.json())
print(data.shape)
data[["flight_number", "date_utc", "static_fire_date_utc", "success"]].head()

200
(107, 42)


,flight_number,date_utc,static_fire_date_utc,success
0,1,2006-03-24T22:30:00.000Z,2006-03-17T00:00:00.000Z,False
1,2,2007-03-21T01:10:00.000Z,NaN,False
2,3,2008-08-03T03:34:00.000Z,NaN,False
3,4,2008-09-28T23:15:00.000Z,2008-09-20T00:00:00.000Z,True
4,5,2009-07-13T03:35:00.000Z,NaN,True


## 2. Load the collected Falcon 9 dataset

In [3]:
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv")
df = df[df["BoosterVersion"] == "Falcon 9"]
print(df.shape)
df.head()

(90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## 3. Missing values

In [4]:
payload_mean = df["PayloadMass"].mean()
df["PayloadMass"] = df["PayloadMass"].fillna(payload_mean)
print(round(payload_mean, 2))
df.isnull().sum()[lambda s: s > 0]

6104.96


LandingPad    26
dtype: int64

`LandingPad` stays empty on purpose: a missing value means no landing pad was used.

## 4. Launches per site, orbit and outcome

In [5]:
df["LaunchSite"].value_counts()

LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

In [6]:
df["Orbit"].value_counts()

Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
ES-L1     1
HEO       1
SO        1
GEO       1
Name: count, dtype: int64

In [7]:
landing_outcomes = df["Outcome"].value_counts()
landing_outcomes

Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64

## 5. Create the target

In [8]:
bad_outcomes = {o for o in landing_outcomes.index if not o.startswith("True")}
print(bad_outcomes)

df["Class"] = [0 if outcome in bad_outcomes else 1 for outcome in df["Outcome"]]
print("Success rate:", round(df["Class"].mean(), 4))

{'False Ocean', 'False ASDS', 'None None', 'None ASDS', 'False RTLS'}
Success rate: 0.6667


In [9]:
df.groupby("LaunchSite")["Class"].agg(["count", "mean"]).round(3)

,count,mean
LaunchSite,,
CCAFS SLC 40,55,0.600
KSC LC 39A,22,0.773
VAFB SLC 4E,13,0.769


In [10]:
df.groupby("Orbit")["Class"].agg(["count", "mean"]).sort_values("count", ascending=False).round(3)

,count,mean
Orbit,,
GTO,27,0.519
ISS,21,0.619
VLEO,14,0.857
PO,9,0.667
LEO,7,0.714
SSO,5,1.000
MEO,3,0.667
GEO,1,1.000
ES-L1,1,1.000
